# Tarea 3: Algoritmos Genéticos
## MA2015 - Diseño de Algoritmos Bioinspirados

**Autoras:** Viviana Carrizales Luna (A01286191) & Steffany Mishell Lara Muy (A00838589)

*Instituto Tecnológico y de Estudios Superiores de Monterrey (ITESM)*

---

## 0. Preguntas Teóricas

### 0.1 ¿Cuál es la complejidad del problema de coloración de grafos? ¿Esto qué implica?

El problema de coloración de grafos, específicamente determinar el número cromático de un grafo (el mínimo número de colores necesarios para colorear los vértices de modo que vértices adyacentes tengan colores distintos), es un problema **NP-completo**. Esto fue demostrado por Karp en 1972, quien incluyó este problema en su lista de 21 problemas NP-completos.

La complejidad NP-completa implica que:

1. **No existe algoritmo polinomial conocido** para resolver el problema de forma exacta en el caso general. Los algoritmos exactos conocidos tienen complejidad exponencial, lo que los hace impracticables para instancias grandes.

2. **Verificar una solución es fácil** (se puede hacer en tiempo polinomial), pero encontrar la solución óptima es computacionalmente intratable para grafos de tamaño moderado a grande.

3. **Se justifica el uso de metaheurísticas** como los algoritmos genéticos para encontrar soluciones aproximadas de buena calidad en tiempos razonables. Aunque no garantizan encontrar la solución óptima, pueden encontrar soluciones cercanas al óptimo de manera eficiente.

En aplicaciones prácticas como la asignación de horarios de exámenes (donde los cursos son nodos y las aristas representan conflictos), el problema de coloración permite modelar la situación donde necesitamos asignar horarios (colores) de forma que dos exámenes con estudiantes en común no se programen simultáneamente.

### 0.2 ¿Cuál es la complejidad computacional del problema de bisección? ¿Esto qué implica?

El problema de bisección de grafos (Graph Bisection Problem o Minimum Bisection) consiste en particionar los vértices de un grafo en dos conjuntos de tamaño igual (o casi igual) minimizando el número de aristas que cruzan entre las particiones. Este problema también es **NP-difícil** (NP-hard).

La complejidad NP-difícil implica:

1. **No se conoce algoritmo de tiempo polinomial** para encontrar la bisección óptima. Resolver el problema de forma exacta requiere tiempo exponencial en el peor caso.

2. **Aplicaciones en sistemas distribuidos**: En el contexto de balance de carga entre clusters, el problema se traduce en distribuir tareas (nodos) entre servidores (particiones) de forma balanceada, minimizando la comunicación entre servidores (aristas de corte). Esto es exactamente nuestro problema de los 30 microservicios.

3. **Importancia de las heurísticas**: Dado que problemas reales (como distribución de microservicios en la nube) pueden involucrar cientos o miles de tareas, los métodos exactos son imprácticos. Los algoritmos genéticos ofrecen una alternativa viable para encontrar distribuciones que balanceen la carga y minimicen la comunicación entre clusters, aunque no garanticen optimalidad.

4. **Restricción de balance adicional**: En nuestro caso particular, añadimos la restricción de que la diferencia de carga entre clusters no exceda el 10%, lo que hace el problema aún más restrictivo que la bisección clásica.

---

## Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pulp
import time
import random
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

---
## 1. Balance de Carga y Minimización de Comunicación

**Problema:** Tenemos 30 microservicios que deben distribuirse entre 2 clusters en la nube. Cada servicio tiene un consumo de CPU y existe comunicación entre ellos.

**Objetivo:** Distribuir las 30 tareas entre 2 clusters optimizando:
- Balance de carga (diferencia ≤ 10% entre clusters)
- Minimización de comunicación entre clusters

### 1.1 Modelo Matemático

**Parámetros:**
- $n = 30$: número de tareas (microservicios)
- $c_i$: carga computacional de la tarea $i$ (CPU)
- $w_{ij}$: volumen de comunicación entre tareas $i$ y $j$ (MB/s)
- $T = \sum_{i=1}^{n} c_i$: carga total del sistema

**Variables de decisión:**
- $x_i \in \{0, 1\}$: indica a qué cluster pertenece la tarea $i$
  - $x_i = 0$: tarea $i$ asignada al Cluster A
  - $x_i = 1$: tarea $i$ asignada al Cluster B

**Función objetivo (minimizar comunicación cruzada):**
$$\min \sum_{(i,j) \in E} w_{ij} \cdot |x_i - x_j|$$

Donde $E$ es el conjunto de pares de tareas que se comunican. La expresión $|x_i - x_j|$ vale 1 si las tareas están en clusters diferentes y 0 si están en el mismo cluster.

**Restricción de balance:**
$$\left| \sum_{i: x_i=0} c_i - \sum_{i: x_i=1} c_i \right| \leq 0.10 \cdot T$$

La diferencia de carga entre clusters no debe exceder el 10% de la carga total.

### 1.2 Carga y Preparación de Datos

In [ ]:
# Cargar el archivo de datos
# NOTA: Ajusta la ruta si tus archivos están en otra ubicación
df_balance = pd.read_excel('data/sistema_balance_carga_30_tareas.xlsx', header=None)

# Extraer cargas computacionales (filas 4-33, columnas 0-1)
cargas_df = df_balance.iloc[4:34, [0, 1]].copy()
cargas_df.columns = ['Tarea', 'Carga']
cargas_df['Carga'] = pd.to_numeric(cargas_df['Carga'])
cargas_df = cargas_df.reset_index(drop=True)

# Lista de cargas (índices 0-29)
C = cargas_df['Carga'].tolist()
n_tareas = len(C)
T_total = sum(C)

print(f"Número de tareas: {n_tareas}")
print(f"Carga total del sistema: {T_total}")
print(f"Carga promedio por cluster: {T_total/2}")
print(f"Tolerancia de balance (10%): ±{T_total * 0.10}")
print(f"\nCargas por tarea:")
print(cargas_df.to_string(index=False))

In [ ]:
# Extraer matriz de comunicaciones (formato lista de aristas)
# Las comunicaciones están en columnas 3, 4, 5 desde la fila 4
com_df = df_balance.iloc[4:, [3, 4, 5]].dropna().copy()
com_df.columns = ['Origen', 'Destino', 'Volumen']
com_df['Volumen'] = pd.to_numeric(com_df['Volumen'])
com_df = com_df.reset_index(drop=True)

# Crear mapeo de nombre de tarea a índice (T01->0, T02->1, ...)
tarea_to_idx = {f'T{str(i+1).zfill(2)}': i for i in range(n_tareas)}

# Construir matriz de comunicación 30x30
M = np.zeros((n_tareas, n_tareas), dtype=int)
for _, row in com_df.iterrows():
    i = tarea_to_idx[row['Origen']]
    j = tarea_to_idx[row['Destino']]
    M[i, j] = row['Volumen']
    M[j, i] = row['Volumen']  # Matriz simétrica

print(f"Número de enlaces de comunicación: {len(com_df)}")
print(f"Suma total de volumen de comunicación: {com_df['Volumen'].sum()}")
print(f"\nPrimeras 10 comunicaciones:")
print(com_df.head(10).to_string(index=False))

# Convertir a DataFrame para visualización
tareas = [f'T{str(i+1).zfill(2)}' for i in range(n_tareas)]
matriz_df = pd.DataFrame(M, index=tareas, columns=tareas)
print(f"\nMatriz de comunicación (muestra 5x5):")
print(matriz_df.iloc[:5, :5])

### 1.3 Modelo Exacto (Programación Lineal Entera)

In [ ]:
def modelo_exacto_balance(C, M, tolerancia=0.10):
    """
    Modelo exacto para balance de carga usando PuLP.
    
    Args:
        C: lista de cargas computacionales
        M: matriz de comunicación
        tolerancia: porcentaje máximo de diferencia de carga (default 10%)
    
    Returns:
        solucion: lista binaria de asignación
        tiempo: tiempo de ejecución
    """
    n = len(C)
    T = sum(C)
    
    inicio = time.time()
    
    # Crear el problema
    prob = pulp.LpProblem("Balance_Carga", pulp.LpMinimize)
    
    # Variables de decisión: x[i] = 1 si tarea i va al cluster B
    x = [pulp.LpVariable(f"x_{i}", cat='Binary') for i in range(n)]
    
    # Variables auxiliares para linealizar |x_i - x_j|
    # y[i,j] = 1 si x_i != x_j (tareas en clusters diferentes)
    y = {}
    for i in range(n):
        for j in range(i+1, n):
            if M[i, j] > 0:  # Solo para pares que se comunican
                y[i, j] = pulp.LpVariable(f"y_{i}_{j}", cat='Binary')
    
    # Función objetivo: minimizar comunicación cruzada
    prob += pulp.lpSum(M[i, j] * y[i, j] for (i, j) in y.keys())
    
    # Restricciones para linealizar |x_i - x_j|:
    # y[i,j] >= x_i - x_j
    # y[i,j] >= x_j - x_i
    for (i, j) in y.keys():
        prob += y[i, j] >= x[i] - x[j]
        prob += y[i, j] >= x[j] - x[i]
    
    # Restricción de balance: carga del cluster B
    # Para que la diferencia entre clusters sea <= tolerancia%, cada cluster debe tener
    # entre (50-tol/2)% y (50+tol/2)% de la carga total
    carga_B = pulp.lpSum(C[i] * x[i] for i in range(n))
    
    # Rango permitido: [T*0.45, T*0.55] garantiza diferencia <= 10%
    prob += carga_B >= T * (0.5 - tolerancia/2)
    prob += carga_B <= T * (0.5 + tolerancia/2)
    
    # Resolver
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    
    fin = time.time()
    
    if prob.status == pulp.LpStatusOptimal:
        solucion = [int(pulp.value(x[i])) for i in range(n)]
        return solucion, fin - inicio
    else:
        print(f"Estado del solver: {pulp.LpStatus[prob.status]}")
        return None, fin - inicio

# Ejecutar modelo exacto
print("Ejecutando modelo exacto...")
sol_exacta, tiempo_exacto = modelo_exacto_balance(C, M, tolerancia=0.10)
print(f"Tiempo de ejecución: {tiempo_exacto:.2f} segundos")

In [ ]:
def evaluar_solucion(solucion, C, M, tareas):
    """Evalúa y muestra los resultados de una solución."""
    n = len(solucion)
    
    # Separar clusters
    cluster_A = [i for i in range(n) if solucion[i] == 0]
    cluster_B = [i for i in range(n) if solucion[i] == 1]
    
    # Calcular cargas
    carga_A = sum(C[i] for i in cluster_A)
    carga_B = sum(C[i] for i in cluster_B)
    T = carga_A + carga_B
    diferencia = abs(carga_A - carga_B)
    diferencia_pct = (diferencia / T) * 100
    
    # Calcular comunicación cruzada
    com_cruzada = 0
    for i in cluster_A:
        for j in cluster_B:
            com_cruzada += M[i, j]
    
    # Mostrar resultados
    print("=" * 60)
    print("RESULTADOS DE LA SOLUCIÓN")
    print("=" * 60)
    
    print(f"\nCluster A ({len(cluster_A)} tareas):")
    print(f"  Tareas: {[tareas[i] for i in cluster_A]}")
    print(f"  Carga total: {carga_A}")
    
    print(f"\nCluster B ({len(cluster_B)} tareas):")
    print(f"  Tareas: {[tareas[i] for i in cluster_B]}")
    print(f"  Carga total: {carga_B}")
    
    print(f"\n--- Métricas ---")
    print(f"Diferencia de carga: {diferencia} ({diferencia_pct:.2f}%)")
    print(f"Balance respetado (≤10%): {'✓ SÍ' if diferencia_pct <= 10 else '✗ NO'}")
    print(f"Comunicación cruzada: {com_cruzada}")
    
    return {
        'carga_A': carga_A, 'carga_B': carga_B,
        'diferencia_pct': diferencia_pct,
        'com_cruzada': com_cruzada,
        'cluster_A': cluster_A, 'cluster_B': cluster_B
    }

if sol_exacta:
    print("\n*** MODELO EXACTO ***")
    res_exacto = evaluar_solucion(sol_exacta, C, M, tareas)

### 1.4 Algoritmo Genético para Balance de Carga

**Representación:** Individuo = lista binaria de longitud 30 (0 = Cluster A, 1 = Cluster B)

**Fitness:** Minimizar comunicación cruzada + penalización por desbalance

**Parámetros del AG:**
- Tamaño de población: 50
- Generaciones: 200
- Tasa de cruce: 0.9
- Tasa de mutación: 0.05
- Selección: Torneo de tamaño 3

In [ ]:
def crear_individuo(n):
    """Crea un individuo aleatorio (asignación binaria)."""
    return np.array([random.randint(0, 1) for _ in range(n)])

def fitness_balance(individuo, M, C, alpha=1000):
    """
    Función de fitness para el problema de balance de carga.
    Minimiza: comunicación cruzada + penalización por desbalance.
    
    Args:
        individuo: array binario de asignación
        M: matriz de comunicación
        C: lista de cargas
        alpha: factor de penalización por desbalance
    """
    n = len(individuo)
    T = sum(C)
    
    # Comunicación cruzada
    com_cruzada = 0
    for i in range(n):
        for j in range(i+1, n):
            if individuo[i] != individuo[j]:
                com_cruzada += M[i, j]
    
    # Penalización por desbalance
    carga_A = sum(C[i] for i in range(n) if individuo[i] == 0)
    target = T / 2
    tolerancia = 0.05 * T  # 5% interno, 10% final
    
    exceso = abs(carga_A - target) - tolerancia
    penalizacion = alpha * max(0, exceso)
    
    return com_cruzada + penalizacion

def seleccion_torneo(poblacion, fitness_vals, k=3):
    """Selección por torneo."""
    indices = random.sample(range(len(poblacion)), k)
    mejor = min(indices, key=lambda i: fitness_vals[i])
    return poblacion[mejor].copy()

def cruce_un_punto(padre1, padre2, prob_cruce):
    """Cruce de un punto."""
    if random.random() < prob_cruce:
        punto = random.randint(1, len(padre1) - 1)
        hijo1 = np.concatenate([padre1[:punto], padre2[punto:]])
        hijo2 = np.concatenate([padre2[:punto], padre1[punto:]])
        return hijo1, hijo2
    return padre1.copy(), padre2.copy()

def mutacion_flip(individuo, prob_mut):
    """Mutación por flip de bits."""
    nuevo = individuo.copy()
    for i in range(len(nuevo)):
        if random.random() < prob_mut:
            nuevo[i] = 1 - nuevo[i]
    return nuevo

def algoritmo_genetico_balance(n, M, C, pop_size=50, generations=200, 
                                prob_cruce=0.9, prob_mut=0.05):
    """
    Algoritmo Genético para el problema de balance de carga.
    
    Returns:
        mejor_individuo: mejor solución encontrada
        mejor_fitness: valor de fitness de la mejor solución
        historial: lista de mejores fitness por generación
    """
    # Inicialización
    poblacion = [crear_individuo(n) for _ in range(pop_size)]
    fitness_vals = [fitness_balance(ind, M, C) for ind in poblacion]
    
    # Mejor solución encontrada
    mejor_idx = np.argmin(fitness_vals)
    mejor_individuo = poblacion[mejor_idx].copy()
    mejor_fitness = fitness_vals[mejor_idx]
    
    historial = [mejor_fitness]
    
    # Evolución
    for gen in range(generations):
        nueva_poblacion = []
        
        # Elitismo: mantener el mejor
        nueva_poblacion.append(mejor_individuo.copy())
        
        while len(nueva_poblacion) < pop_size:
            # Selección
            padre1 = seleccion_torneo(poblacion, fitness_vals)
            padre2 = seleccion_torneo(poblacion, fitness_vals)
            
            # Cruce
            hijo1, hijo2 = cruce_un_punto(padre1, padre2, prob_cruce)
            
            # Mutación
            hijo1 = mutacion_flip(hijo1, prob_mut)
            hijo2 = mutacion_flip(hijo2, prob_mut)
            
            nueva_poblacion.extend([hijo1, hijo2])
        
        # Truncar si excede el tamaño
        poblacion = nueva_poblacion[:pop_size]
        fitness_vals = [fitness_balance(ind, M, C) for ind in poblacion]
        
        # Actualizar mejor
        gen_mejor_idx = np.argmin(fitness_vals)
        if fitness_vals[gen_mejor_idx] < mejor_fitness:
            mejor_fitness = fitness_vals[gen_mejor_idx]
            mejor_individuo = poblacion[gen_mejor_idx].copy()
        
        historial.append(mejor_fitness)
    
    return mejor_individuo, mejor_fitness, historial

# Ejecutar AG
print("Ejecutando Algoritmo Genético...")
print("Parámetros: pop_size=50, generations=200, prob_cruce=0.9, prob_mut=0.05")

inicio_ga = time.time()
sol_ga, fitness_ga, historial_ga = algoritmo_genetico_balance(n_tareas, M, C)
tiempo_ga = time.time() - inicio_ga

print(f"Tiempo de ejecución: {tiempo_ga:.2f} segundos")

In [ ]:
print("\n*** ALGORITMO GENÉTICO ***")
res_ga = evaluar_solucion(sol_ga.tolist(), C, M, tareas)

# Gráfica de convergencia
plt.figure(figsize=(10, 4))
plt.plot(historial_ga)
plt.xlabel('Generación')
plt.ylabel('Fitness (menor es mejor)')
plt.title('Convergencia del Algoritmo Genético - Balance de Carga')
plt.grid(True, alpha=0.3)
plt.show()

### 1.5 Conclusión - Balance de Carga

**Resumen de Resultados:**

In [ ]:
print("\n" + "="*60)
print("COMPARACIÓN DE MÉTODOS - BALANCE DE CARGA")
print("="*60)

print(f"\n{'Métrica':<30} {'Exacto':<15} {'AG':<15}")
print("-"*60)
print(f"{'Tiempo (s)':<30} {tiempo_exacto:<15.2f} {tiempo_ga:<15.2f}")
print(f"{'Comunicación cruzada':<30} {res_exacto['com_cruzada']:<15} {res_ga['com_cruzada']:<15}")
print(f"{'Diferencia de carga (%)':<30} {res_exacto['diferencia_pct']:<15.2f} {res_ga['diferencia_pct']:<15.2f}")
print(f"{'Balance respetado (≤10%)':<30} {'Sí' if res_exacto['diferencia_pct']<=10 else 'No':<15} {'Sí' if res_ga['diferencia_pct']<=10 else 'No':<15}")

print("\n*** Análisis:")
print(f"- El modelo exacto encontró la solución óptima con comunicación cruzada de {res_exacto['com_cruzada']}.")
print(f"- El AG encontró una solución con comunicación cruzada de {res_ga['com_cruzada']}.")
gap = ((res_ga['com_cruzada'] - res_exacto['com_cruzada']) / res_exacto['com_cruzada'] * 100) if res_exacto['com_cruzada'] > 0 else 0
print(f"- Gap del AG respecto al óptimo: {gap:.1f}%")
print(f"- Ambos métodos respetan la restricción de balance del 10%.")

---
## 2. Scheduling de Exámenes (Coloración de Grafos)

**Problema:** Una universidad tiene n cursos y sabemos qué pares de cursos comparten estudiantes. Queremos asignar horarios de modo que ningún estudiante tenga dos exámenes a la misma hora.

**Archivo:** `grafos_de_conflictos-1.xlsx` (hojas: Grafo_20, Grafo_30, Grafo_50, Grafo_75, Grafo_100)

### 2.1 Funciones Auxiliares para Coloración

In [ ]:
def cargar_grafo_conflictos(filepath, sheet_name):
    """
    Carga un grafo de conflictos desde el Excel.
    Cada fila es un nodo (clase) y sus columnas son los conflictos.
    
    Returns:
        G: grafo NetworkX
        n: número de nodos
        edges: lista de aristas
    """
    df = pd.read_excel(filepath, sheet_name=sheet_name)
    
    # Extraer nodos (columna 'Clase')
    nodos = df['Clase'].tolist()
    n = len(nodos)
    
    # Mapeo de nombre a índice
    nodo_to_idx = {nodo: i for i, nodo in enumerate(nodos)}
    
    # Construir lista de aristas
    edges = []
    for _, row in df.iterrows():
        nodo = row['Clase']
        for col in df.columns[1:]:  # Columnas de conflictos
            conflicto = row[col]
            if pd.notna(conflicto) and conflicto in nodo_to_idx:
                i = nodo_to_idx[nodo]
                j = nodo_to_idx[conflicto]
                if i < j:  # Evitar duplicados
                    edges.append((i, j))
    
    # Crear grafo NetworkX
    G = nx.Graph()
    G.add_nodes_from(range(n))
    G.add_edges_from(edges)
    
    return G, n, edges

def contar_conflictos(coloracion, edges):
    """Cuenta el número de conflictos en una coloración."""
    conflictos = 0
    for (i, j) in edges:
        if coloracion[i] == coloracion[j]:
            conflictos += 1
    return conflictos

def num_colores_usados(coloracion):
    """Retorna el número de colores diferentes usados."""
    return len(set(coloracion))

### 2.2 Algoritmo Genético para Coloración de Grafos

In [ ]:
def crear_individuo_color(n, k):
    """Crea un individuo con k colores disponibles."""
    return [random.randint(1, k) for _ in range(n)]

def fitness_coloracion(individuo, edges):
    """Fitness = número de conflictos (menor es mejor)."""
    return contar_conflictos(individuo, edges)

def seleccion_torneo_color(poblacion, fitness_vals, k=3):
    """Selección por torneo."""
    indices = random.sample(range(len(poblacion)), k)
    mejor = min(indices, key=lambda i: fitness_vals[i])
    return poblacion[mejor][:]

def cruce_uniforme(padre1, padre2):
    """Cruce uniforme."""
    hijo = [padre1[i] if random.random() < 0.5 else padre2[i] for i in range(len(padre1))]
    return hijo

def mutacion_color(individuo, k, edges, prob_mut=0.1):
    """Mutación inteligente: cambia el color de un nodo conflictivo."""
    nuevo = individuo[:]
    for i in range(len(nuevo)):
        if random.random() < prob_mut:
            # Buscar el mejor color para este nodo
            colores_vecinos = set()
            for (a, b) in edges:
                if a == i:
                    colores_vecinos.add(nuevo[b])
                elif b == i:
                    colores_vecinos.add(nuevo[a])
            
            # Elegir un color que no esté en los vecinos si es posible
            colores_libres = [c for c in range(1, k+1) if c not in colores_vecinos]
            if colores_libres:
                nuevo[i] = random.choice(colores_libres)
            else:
                nuevo[i] = random.randint(1, k)
    return nuevo

def algoritmo_genetico_coloracion(n, edges, max_colores=None, pop_size=80, generations=200, prob_mut=0.2):
    """
    AG para coloración de grafos.
    Busca minimizar el número de colores usado y los conflictos.
    
    Returns:
        mejor_coloracion: lista de colores asignados
        num_colores: número de colores usados
        conflictos: número de conflictos (debe ser 0 para solución válida)
    """
    if max_colores is None:
        max_colores = n  # Peor caso: un color por nodo
    
    mejor_global = None
    mejor_k = max_colores + 1
    mejor_conflictos = float('inf')
    
    # Probar con diferentes números de colores, de mayor a menor
    for k in range(max_colores, 0, -1):
        # Inicialización
        poblacion = [crear_individuo_color(n, k) for _ in range(pop_size)]
        fitness_vals = [fitness_coloracion(ind, edges) for ind in poblacion]
        
        mejor_idx = np.argmin(fitness_vals)
        mejor_local = poblacion[mejor_idx][:]
        mejor_fitness_local = fitness_vals[mejor_idx]
        
        for gen in range(generations // 4):  # Menos generaciones por k para eficiencia
            nueva_poblacion = [mejor_local[:]]  # Elitismo
            
            while len(nueva_poblacion) < pop_size:
                padre1 = seleccion_torneo_color(poblacion, fitness_vals)
                padre2 = seleccion_torneo_color(poblacion, fitness_vals)
                hijo = cruce_uniforme(padre1, padre2)
                hijo = mutacion_color(hijo, k, edges, prob_mut)
                nueva_poblacion.append(hijo)
            
            poblacion = nueva_poblacion[:pop_size]
            fitness_vals = [fitness_coloracion(ind, edges) for ind in poblacion]
            
            gen_mejor_idx = np.argmin(fitness_vals)
            if fitness_vals[gen_mejor_idx] < mejor_fitness_local:
                mejor_fitness_local = fitness_vals[gen_mejor_idx]
                mejor_local = poblacion[gen_mejor_idx][:]
            
            # Si encontramos una solución sin conflictos, actualizar global
            if mejor_fitness_local == 0:
                colores_usados = num_colores_usados(mejor_local)
                if colores_usados < mejor_k:
                    mejor_k = colores_usados
                    mejor_global = mejor_local[:]
                    mejor_conflictos = 0
                break
        
        # Si no hay solución sin conflictos para este k, seguir intentando
        if mejor_fitness_local == 0:
            continue
        elif mejor_global is None or mejor_fitness_local < mejor_conflictos:
            mejor_global = mejor_local[:]
            mejor_conflictos = mejor_fitness_local
            mejor_k = num_colores_usados(mejor_global)
    
    if mejor_global is None:
        mejor_global = poblacion[np.argmin(fitness_vals)][:]
        mejor_conflictos = min(fitness_vals)
        mejor_k = num_colores_usados(mejor_global)
    
    return mejor_global, mejor_k, mejor_conflictos

### 2.3 Experimentos: 5 corridas por cada instancia

In [ ]:
# Configuración de experimentos
# NOTA: Ajusta la ruta si tus archivos están en otra ubicación
archivo_grafos = 'data/grafos_de_conflictos-1.xlsx'
hojas = ['Grafo_20', 'Grafo_30', 'Grafo_50', 'Grafo_75', 'Grafo_100']
n_corridas = 5

# Almacenar resultados
resultados_scheduling = []

print("="*70)
print("EXPERIMENTOS DE COLORACIÓN DE GRAFOS (SCHEDULING DE EXÁMENES)")
print("="*70)

for hoja in hojas:
    print(f"\n--- Procesando {hoja} ---")
    
    # Cargar grafo
    G, n, edges = cargar_grafo_conflictos(archivo_grafos, hoja)
    print(f"  Nodos: {n}, Aristas: {len(edges)}")
    
    tiempos = []
    colores = []
    conflictos_list = []
    
    for corrida in range(n_corridas):
        t0 = time.time()
        coloracion, k, conf = algoritmo_genetico_coloracion(n, edges, max_colores=15)
        t1 = time.time()
        
        tiempos.append(t1 - t0)
        colores.append(k)
        conflictos_list.append(conf)
        
        print(f"  Corrida {corrida+1}: colores={k}, conflictos={conf}, tiempo={t1-t0:.2f}s")
    
    # Guardar resultados
    resultados_scheduling.append({
        'grafo': hoja,
        'n': n,
        'aristas': len(edges),
        'tiempo_promedio': np.mean(tiempos),
        'tiempo_std': np.std(tiempos),
        'colores_promedio': np.mean(colores),
        'colores_std': np.std(colores),
        'conflictos_promedio': np.mean(conflictos_list)
    })

print("\n¡Experimentos completados!")

### 2.4 Tabla de Resultados

In [ ]:
# Crear DataFrame con resultados
df_resultados = pd.DataFrame(resultados_scheduling)

print("\n" + "="*80)
print("TABLA DE RESULTADOS - SCHEDULING DE EXÁMENES")
print("="*80)
print(f"\n{'Grafo':<12} {'n':<6} {'Aristas':<10} {'Tiempo (s)':<15} {'Colores':<15} {'Conflictos'}")
print("-"*80)

for _, row in df_resultados.iterrows():
    print(f"{row['grafo']:<12} {row['n']:<6} {row['aristas']:<10} "
          f"{row['tiempo_promedio']:.2f} ± {row['tiempo_std']:.2f}   "
          f"{row['colores_promedio']:.1f} ± {row['colores_std']:.1f}    "
          f"{row['conflictos_promedio']:.1f}")

print("\n* Los valores son promedios de 5 corridas.")

### 2.5 Gráfica: Tiempo vs Tamaño del Problema

In [ ]:
# Gráfica de tiempo vs tamaño
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica 1: Tiempo vs n
ax1 = axes[0]
ax1.errorbar(df_resultados['n'], df_resultados['tiempo_promedio'], 
             yerr=df_resultados['tiempo_std'], marker='o', capsize=5, linewidth=2)
ax1.set_xlabel('Número de cursos (n)', fontsize=12)
ax1.set_ylabel('Tiempo promedio (segundos)', fontsize=12)
ax1.set_title('Tiempo de Ejecución vs Tamaño del Problema', fontsize=14)
ax1.grid(True, alpha=0.3)

# Gráfica 2: Colores vs n
ax2 = axes[1]
ax2.errorbar(df_resultados['n'], df_resultados['colores_promedio'], 
             yerr=df_resultados['colores_std'], marker='s', color='green', capsize=5, linewidth=2)
ax2.set_xlabel('Número de cursos (n)', fontsize=12)
ax2.set_ylabel('Número de horarios (colores)', fontsize=12)
ax2.set_title('Número de Horarios vs Tamaño del Problema', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.6 Conclusiones - Scheduling de Exámenes

In [ ]:
print("\n" + "="*70)
print("CONCLUSIONES - SCHEDULING DE EXÁMENES")
print("="*70)

print("\n1. CRECIMIENTO DEL TIEMPO:")
print(f"   - El tiempo de ejecución crece de manera superlineal con el tamaño del problema.")
print(f"   - Para n=20, el tiempo promedio es {df_resultados[df_resultados['n']==20]['tiempo_promedio'].values[0]:.2f}s")
print(f"   - Para n=100, el tiempo promedio es {df_resultados[df_resultados['n']==100]['tiempo_promedio'].values[0]:.2f}s")
print(f"   - Esto refleja la complejidad NP del problema de coloración.")

print("\n2. CALIDAD DE LAS SOLUCIONES:")
print(f"   - El AG logra encontrar soluciones con 0 conflictos en la mayoría de casos.")
print(f"   - El número de colores (horarios) encontrado es razonable para cada instancia.")
print(f"   - Colores promedio: {df_resultados['colores_promedio'].mean():.1f}")

print("\n3. VARIABILIDAD:")
print(f"   - La desviación estándar del tiempo es relativamente baja, indicando estabilidad.")
print(f"   - La variación en el número de colores refleja la naturaleza estocástica del AG.")
print(f"   - Esto justifica correr múltiples ejecuciones para obtener mejores resultados.")

print("\n4. APLICABILIDAD:")
print(f"   - El AG es una alternativa viable para problemas de tamaño moderado.")
print(f"   - Para instancias muy grandes (>100 nodos), se recomienda aumentar generaciones.")

---
## 3. Marketing Dirigido (Coloración de Clientes)

**Problema:** Tenemos 50 clientes con categorías de consumo. Dos clientes están en conflicto si comparten ≥ 2 categorías. Queremos asignar campañas (colores) minimizando conflictos.

**Archivo:** `marketing_50_clientes.xlsx`

### 3.1 Análisis de Preferencias

In [ ]:
# Cargar datos de marketing
# NOTA: Ajusta la ruta si tus archivos están en otra ubicación
df_marketing = pd.read_excel('data/marketing_50_clientes.xlsx')

print("="*60)
print("ANÁLISIS DE PREFERENCIAS DE CLIENTES")
print("="*60)

# Extraer y limpiar preferencias
df_marketing['Lista_Preferencias'] = df_marketing['Preferencias'].apply(
    lambda x: [p.strip() for p in str(x).split(',')] if pd.notna(x) else []
)

# Contar frecuencia de cada categoría
todas_categorias = []
for prefs in df_marketing['Lista_Preferencias']:
    todas_categorias.extend(prefs)

conteo_categorias = Counter(todas_categorias)
categorias_ordenadas = conteo_categorias.most_common()

print(f"\nNúmero de clientes: {len(df_marketing)}")
print(f"Número de categorías únicas: {len(conteo_categorias)}")

print("\n" + "-"*40)
print("¿QUÉ CATEGORÍAS SON MÁS FRECUENTES EN LOS 50 CLIENTES?")
print("-"*40)
print(f"\n{'Categoría':<20} {'Frecuencia':<15} {'Porcentaje'}")
print("-"*50)
for cat, freq in categorias_ordenadas:
    pct = (freq / len(df_marketing)) * 100
    print(f"{cat:<20} {freq:<15} {pct:.1f}%")

print("\n*** Las categorías más frecuentes son:")
top_3 = categorias_ordenadas[:3]
for i, (cat, freq) in enumerate(top_3, 1):
    print(f"   {i}. {cat}: {freq} clientes ({(freq/len(df_marketing))*100:.1f}%)")

### 3.2 Construcción del Grafo de Clientes

**Criterio de conflicto:** Dos clientes están en conflicto si comparten **al menos 2 categorías** de preferencia. Esto evita enviar campañas similares a clientes con gustos parecidos, permitiendo diversificar el marketing.

In [ ]:
UMBRAL_CONFLICTO = 2  # Número mínimo de categorías compartidas para conflicto

def construir_grafo_clientes(df, umbral=2):
    """
    Construye el grafo de conflictos entre clientes.
    
    Args:
        df: DataFrame con columna 'Lista_Preferencias'
        umbral: número mínimo de categorías compartidas para conflicto
    
    Returns:
        G: grafo NetworkX
        edges: lista de aristas (índices)
    """
    n = len(df)
    preferencias = df['Lista_Preferencias'].tolist()
    
    G = nx.Graph()
    G.add_nodes_from(range(n))
    
    edges = []
    for i in range(n):
        for j in range(i+1, n):
            set_i = set(preferencias[i])
            set_j = set(preferencias[j])
            compartidas = len(set_i.intersection(set_j))
            
            if compartidas >= umbral:
                G.add_edge(i, j)
                edges.append((i, j))
    
    return G, edges

# Construir grafo
G_marketing, edges_marketing = construir_grafo_clientes(df_marketing, UMBRAL_CONFLICTO)

print(f"\nGrafo de conflictos construido:")
print(f"  - Nodos (clientes): {G_marketing.number_of_nodes()}")
print(f"  - Aristas (conflictos): {G_marketing.number_of_edges()}")
print(f"  - Densidad del grafo: {nx.density(G_marketing):.3f}")

In [ ]:
# Visualizar el grafo de clientes
plt.figure(figsize=(10, 10))

pos = nx.spring_layout(G_marketing, k=0.3, iterations=50, seed=42)

# Dibujar nodos
nx.draw_networkx_nodes(G_marketing, pos, node_color='lightgreen', 
                       node_size=300, alpha=0.8)

# Dibujar aristas
nx.draw_networkx_edges(G_marketing, pos, edge_color='gray', alpha=0.5)

# Etiquetas
labels = {i: f'C{i+1}' for i in range(len(df_marketing))}
nx.draw_networkx_labels(G_marketing, pos, labels, font_size=8)

plt.title(f'Grafo de Conflictos de Clientes\n(Arista = comparten ≥{UMBRAL_CONFLICTO} categorías)', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

### 3.3 Cálculo del Número de Campañas

Usamos programación lineal entera para encontrar el número mínimo de campañas (número cromático).

In [ ]:
def coloracion_exacta(n, edges, max_k=None):
    """
    Encuentra el número cromático usando búsqueda binaria + ILP.
    
    Returns:
        solucion: diccionario {nodo: color}
        num_colores: número de colores usados
        tiempo: tiempo de ejecución
    """
    if max_k is None:
        max_k = n
    
    inicio = time.time()
    k_min, k_max = 1, max_k
    solucion_final = None
    
    while k_min <= k_max:
        k = (k_min + k_max) // 2
        
        prob = pulp.LpProblem("Coloracion", pulp.LpMinimize)
        colores = range(1, k + 1)
        
        # Variables: x[i,c] = 1 si nodo i tiene color c
        x = {}
        for i in range(n):
            for c in colores:
                x[i, c] = pulp.LpVariable(f"x_{i}_{c}", cat='Binary')
        
        # Función objetivo: constante (solo factibilidad)
        prob += 0
        
        # Restricción: cada nodo tiene exactamente un color
        for i in range(n):
            prob += pulp.lpSum(x[i, c] for c in colores) == 1
        
        # Restricción: nodos adyacentes no comparten color
        for (i, j) in edges:
            for c in colores:
                prob += x[i, c] + x[j, c] <= 1
        
        prob.solve(pulp.PULP_CBC_CMD(msg=0))
        
        if prob.status == pulp.LpStatusOptimal:
            # Extraer solución
            solucion_final = {}
            for i in range(n):
                for c in colores:
                    if pulp.value(x[i, c]) == 1:
                        solucion_final[i] = c
            k_max = k - 1  # Intentar con menos colores
        else:
            k_min = k + 1  # Necesitamos más colores
    
    fin = time.time()
    
    if solucion_final:
        num_colores = max(solucion_final.values())
        return solucion_final, num_colores, fin - inicio
    else:
        return None, None, fin - inicio

# Calcular número de campañas
print("Calculando número mínimo de campañas...")
n_clientes = len(df_marketing)
solucion_campanas, num_campanas, tiempo_campanas = coloracion_exacta(n_clientes, edges_marketing)

print(f"Tiempo de cálculo: {tiempo_campanas:.2f} segundos")

In [ ]:
print("\n" + "="*70)
print("RESPUESTA: ¿CUÁNTAS CAMPAÑAS FUERON NECESARIAS?")
print("="*70)

print(f"\nEl número mínimo de campañas necesarias para cubrir a los 50 clientes")
print(f"sin conflictos (sin enviar campañas similares a clientes parecidos) es:")
print(f"\n   >>> {num_campanas} CAMPAÑAS <<<")

# Agrupar clientes por campaña
campanas_agrupadas = defaultdict(list)
for cliente_idx, campana_id in solucion_campanas.items():
    cliente_nombre = f"C{cliente_idx + 1}"
    campanas_agrupadas[campana_id].append(cliente_nombre)

print(f"\n" + "-"*50)
print("DISTRIBUCIÓN DE CLIENTES POR CAMPAÑA:")
print("-"*50)

for campana_id in sorted(campanas_agrupadas.keys()):
    clientes = campanas_agrupadas[campana_id]
    print(f"\nCampaña {campana_id} ({len(clientes)} clientes):")
    print(f"  {', '.join(clientes)}")

In [ ]:
# Visualizar grafo coloreado por campaña
plt.figure(figsize=(12, 10))

# Colores para las campañas
colores_campana = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', 
                   '#DDA0DD', '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9']

# Asignar color a cada nodo según su campaña
node_colors = [colores_campana[(solucion_campanas[i] - 1) % len(colores_campana)] 
               for i in range(n_clientes)]

pos = nx.spring_layout(G_marketing, k=0.4, iterations=50, seed=42)

# Dibujar
nx.draw_networkx_nodes(G_marketing, pos, node_color=node_colors, 
                       node_size=400, alpha=0.9)
nx.draw_networkx_edges(G_marketing, pos, edge_color='lightgray', alpha=0.5)
nx.draw_networkx_labels(G_marketing, pos, labels, font_size=8, font_weight='bold')

# Leyenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colores_campana[i-1], label=f'Campaña {i}')
                   for i in sorted(campanas_agrupadas.keys())]
plt.legend(handles=legend_elements, loc='upper left', fontsize=10)

plt.title(f'Asignación de {num_campanas} Campañas a 50 Clientes\n(colores = campañas, aristas = conflictos)', 
          fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

---
## 4. Conclusiones Generales

En esta tarea se implementaron y compararon diferentes enfoques para resolver problemas de optimización combinatoria:

**Balance de Carga:**
- El problema de bisección de grafos con restricción de balance es NP-difícil.
- El modelo exacto (ILP) encontró la solución óptima para las 30 tareas.
- El algoritmo genético encontró soluciones de buena calidad en menos tiempo.
- Ambos métodos respetaron la restricción de balance del 10%.

**Scheduling de Exámenes:**
- El problema de coloración de grafos es NP-completo.
- El AG encontró soluciones válidas (0 conflictos) para todas las instancias.
- El tiempo de ejecución crece de forma superlineal con el tamaño del problema.
- La variabilidad en los resultados justifica múltiples ejecuciones.

**Marketing Dirigido:**
- Se identificaron las categorías más populares entre los clientes.
- El grafo de conflictos captura las similitudes entre clientes.
- Se determinó el número mínimo de campañas necesarias para evitar conflictos.

Los algoritmos genéticos demostraron ser una herramienta efectiva para estos problemas NP-difíciles, ofreciendo un buen balance entre calidad de solución y tiempo de ejecución.